# 原报告毛利率同比：固定配对训练与回测
单字段毛利率同比百分点增量；不混非GAAP、不剔除一次性费用。共享标记，保留强参考和全部成本，条件性日期缓冲不作首发认证。

In [1]:
from pathlib import Path
from dataclasses import asdict
from copy import deepcopy
import os,json,hashlib
import numpy as np,pandas as pd,torch,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _dataset,_auxiliary_dataset,_prediction_dates,_ensemble
from src.model.minute_pit_allocator import TARGETS
from src.model.tangent_allocator import TangentHeadAllocator,audit_zero_extended_heads
from src.data.delayed_gross_margin import extract_gross_margin,delayed_margin_events,align_margin_events,FEATURE,FLAG
from src.data import QlibProviderConfig,build_qlib_store_from_frame,load_qlib_feature_frame
from qlib.data import D
from src.workflow.pit_constituent_auxiliary_training_allocation import _smooth_weights
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,annual_paired_metrics,stationary_bootstrap_excess
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/delayed_gross_margin_training_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==p['release_sha256']
training,ta=_source(root,p['training_source'],['inputs/daily_frame','inputs/features','inputs/common','inputs/timing','inputs/auxiliary','inputs/audit',*[f'models/base_outer_{seed}' for seed in p['seeds']]])
source,sa=_source(root,p['source'],['inputs/frame']);controller,ca=_source(root,p['price_source'],['inputs/controller','inputs/plan']);action,aa=_source(root,p['action_source'],['inputs/plan','inputs/market','controller/source_recovery_selection']);strong,sta=_source(root,p['strong_source'],['inputs/plan',*[f'analysis/native/{family}_{cost}' for family in ['candidate','control','soxx'] for cost in p['slippage_bps']]])
for key in ['initial_cash','tariff','start','end','as_of','slippage_bps','maximum_soxl_weight','minimum_cash_weight','maximum_drawdown']:assert strong['inputs/plan'][key]==p[key]
assert controller['inputs/plan']['tariff']==p['tariff']
original=training['models/base_outer_71'];base=training['inputs/daily_frame']
filings,fa=_source(root,p['filing_source'],['analysis/verified_facts','inputs/web_text_bytes','inputs/plan']);assert p['source_gate'].startswith('Conditional')
reports=filings['analysis/verified_facts'].copy();case_map={x['quarter']:x for x in filings['inputs/plan']['cases']};margins={}
for row in reports.itertuples():
 case=case_map[row.quarter];payload=filings['inputs/web_text_bytes'][case['file']];assert hashlib.sha256(payload).hexdigest()==case['sha256'];fact=extract_gross_margin(payload.decode(),quarter=row.quarter,accession=row.accession);assert fact['current_eps']==row.current_eps;margins[row.quarter]=fact['gross_margin_pct']
reports['gross_margin_pct']=reports.quarter.map(margins)

qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us');calendar=pd.DatetimeIndex(D.calendar(start_time='2010-03-11',end_time=p['as_of'],freq='day'));events=delayed_margin_events(reports,calendar);features=align_margin_events(events,base.index,base.trade_date)
feature_provider=root/'data/qlib/delayed_gross_margin_20261015';payload=features.copy();payload.insert(0,'datetime',features.index);payload['instrument']='SOXL_SOXX_CASH';payload=payload.reset_index(drop=True);manifest=build_qlib_store_from_frame(payload,provider_uri=feature_provider,freq='day',region='us',as_of=p['as_of'],field_map={'$'+k:k for k in features},source='48 original-accession quarterly gross-margin rows; conditional date-buffered research; source identities in frozen plan')
read=load_qlib_feature_frame(QlibProviderConfig(feature_provider),instruments=['SOXL_SOXX_CASH'],fields=['$'+FEATURE,'$'+FLAG],start=str(base.index.min().date()),end=p['as_of']).xs('SOXL_SOXX_CASH',level='instrument').rename(columns=lambda x:x.removeprefix('$')).astype(float);assert read.index.equals(features.index);np.testing.assert_allclose(read,features,rtol=1e-6,atol=1e-7,equal_nan=True)
features=read;dataset=_dataset(training['inputs/features'].join(features),base[list(TARGETS)],training['inputs/common'],training['inputs/timing'],{'train':(original.spec.train_start,str(original.train_end.date())),'predict':(p['start'],p['end'])});auxiliary=_auxiliary_dataset(training['inputs/auxiliary']['outer'],str(original.train_end.date()))
state_path=card/'run_state.json';digest=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest();state=json.loads(state_path.read_text()) if state_path.exists() else {'plan_sha256':digest,'fits':[],'native':[]};assert state['plan_sha256']==digest
kwargs={'experiment_name':card.name,'uri':str(card/'qlib_recorder')}
if state.get('recorder_id'):kwargs.update(recorder_id=state['recorder_id'],resume=True)
predictions={};audits=[];selection=[]
with R.start(**kwargs):
 rec=R.get_recorder();state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(state,state_path);rec.save_objects(artifact_path='inputs',plan=p,source_audits={'training':ta,'source':sa,'controller':ca,'action':aa,'strong':sta,'filings':fa},events=events,margin_reports=reports,features=features,feature_manifest=manifest);rec.save_objects(artifact_path='sources',feature=(root/'src/data/delayed_gross_margin.py').read_text(),clock=(root/'src/data/delayed_seasonal_eps.py').read_text(),model=(root/'src/model/tangent_allocator.py').read_text(),base_model=(root/'src/model/minute_pit_allocator.py').read_text())
 for seed in p['seeds']:
  template=training[f'models/base_outer_{seed}'];models={f:TangentHeadAllocator(base_features=template.base_features+[FLAG],extra_features=[] if f=='control' else [FEATURE],feature_masks={**template.feature_masks,FEATURE:FLAG},spec={**asdict(template.spec),'root':str(root)},seed=seed,cutoff=str(template.cutoff),train_end=str(template.train_end.date()),minimum_observations=template.minimum_observations) for f in ['control','candidate']};initial=audit_zero_extended_heads(models['control']._initialize(),models['candidate']._initialize(),shared_primary_count=len(template.base_features)+1,extra_count=1)
  for family,model in models.items():
   name=f'{family}_{seed}'
   if name in state['fits']:model=rec.load_object('models/'+name);models[family]=model
   else:
    assert len(state['fits'])<p['maximum_model_fits'];model.fit(dataset,auxiliary_dataset=auxiliary);rec.save_objects(artifact_path='models',**{name:model});state['fits'].append(name);atomic_json(state,state_path)
   predictions[(family,seed)]=_prediction_dates(model.predict(dataset));audits.append({'family':family,'seed':seed,'initialization':initial,**model.audit});rec.log_metrics(completed_fits=len(state['fits']))
  a,b=models['control'],models['candidate'];assert a.primary_scaler==b.primary_scaler and a.common_scaler==b.common_scaler
  for key in ['training_rows','first_training_signal','last_training_signal','last_mature_label_at','auxiliary_rows','epochs']:assert a.audit[key]==b.audit[key]
  selection.append({'seed':seed,**b.extra_scaler[FEATURE]})
 rec.save_objects(artifact_path='analysis',predictions=predictions,training=audits,feature_usage=selection)
frame=source['inputs/frame'];frame=frame.loc[frame.signal_date.between(p['start'],p['end'])].copy();market=action['inputs/market'];dates=pd.DatetimeIndex(market.signal_date);returns=market.set_index('signal_date')[['soxl_return','soxx_return']];targets={f:_smooth_weights(_ensemble(predictions,f,p['seeds']).reindex(dates),spec=original.spec,candidate=original.candidate,asset_returns=returns) for f in ['control','candidate']}
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);books={}
for family in ['control','candidate']:
 local=frame.copy();aligned=targets[family].reindex(pd.DatetimeIndex(frame.signal_date));assert aligned.notna().all().all()
 for col in aligned:local[col]=aligned[col].to_numpy()
 for cost in p['slippage_bps']:
  key=f'{family}_{cost}'
  if key in state['native']:book=rec.load_object('analysis/native/'+key)
  else:
   assert len(state['native'])<p['maximum_new_native_cases'];replay=replay_full_path(local,models=None,plan=p,slippage_bps=cost,controller=controller['inputs/controller'],recovery_spec=recovery,action_spec=spec,selection=action['controller/source_recovery_selection']);book=native_tiered_stage1(local,replay,provider_uri=provider,tariff=p['tariff'],slippage_bps=cost,initial_cash=p['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);book['targets']=replay['targets'];rec.save_objects(artifact_path='analysis/native',**{key:book});state['native'].append(key);atomic_json(state,state_path)
  books[(family,cost)]=book;rec.log_metrics(completed_native_cases=len(state['native']))
for cost in p['slippage_bps']:
 for family,key in [('full_history_tangent','candidate'),('full_history_original','control'),('soxx','soxx')]:books[(family,cost)]=strong[f'analysis/native/{key}_{cost}'];rec.save_objects(artifact_path='analysis/native',**{f'{family}_{cost}':books[(family,cost)]})
rows=[];annual=[];uncertainty=[]
for (family,cost),book in books.items():
 path=book['authority'];b=books[('soxx',cost)]['authority'];assert path.trade_date.equals(b.trade_date);rel=np.log1p(path.strategy_return)-np.log1p(b.strategy_return);rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,b),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'average_cash_pct':100*path.ending_cash_weight.mean(),'positive_126d_fraction':rel.rolling(126).sum().dropna().gt(0).mean()});annual.append(annual_paired_metrics(path,b).assign(family=family,slippage_bps=cost))
 if family=='candidate':
  for comparator in ['control','soxx','full_history_tangent']:uncertainty.append({'slippage_bps':cost,'comparator':comparator,**stationary_bootstrap_excess(path,books[(comparator,cost)]['authority'],mean_block=63,resamples=2000,alpha=.025,seed=20261015)})
events.to_csv(card/'event_scores.csv',index=False)
table=pd.DataFrame(rows);annual=pd.concat(annual,ignore_index=True);uncertainty=pd.DataFrame(uncertainty);cnd=table[table.family.eq('candidate')].set_index('slippage_bps');ctl=table[table.family.eq('control')].set_index('slippage_bps');stronger=table[table.family.eq('full_history_tangent')].set_index('slippage_bps');gates={'beat_paired_control_all_costs':bool(cnd.terminal_wealth.gt(ctl.terminal_wealth+1e-8).all()),'beat_strong_all_costs':bool(cnd.terminal_wealth.gt(stronger.terminal_wealth+1e-8).all()),'net_soxx_excess_all_costs':bool(cnd.relative_wealth.gt(1).all()),'drawdown_30pct':bool(cnd.max_drawdown_pct.ge(-30).all() and cnd.monitored_drawdown_pct.ge(-30).all()),'cvar_no_worse':bool(cnd.cvar5_pct.ge(cnd.benchmark_cvar5_pct).all()),'rolling_70pct':bool(cnd.positive_126d_fraction.ge(.7).all()),'annual_70pct':bool(annual[annual.family.eq('candidate')].groupby('slippage_bps').relative_wealth.apply(lambda x:x.gt(1).mean()).ge(.7).all()),'bootstrap_positive':bool(uncertainty.lower_bound_positive.all()),'independent_validation':False}
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','sources/*','analysis/predictions','analysis/native/*']};summary={'experiment':card.name,'decision':'needs_independent_validation' if all(v for k,v in gates.items() if k!='independent_validation') else p['failure_decision'],'gates':gates,'new_fits':len(state['fits']),'new_native_cases':len(state['native']),'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False,'plan_sha256':digest,'feature_provider_uri':str(feature_provider),'conditional_source_timing':True,'production_qualified':False};rec.save_objects(artifact_path='analysis',summary=summary,comparison=table,annual=annual,uncertainty=uncertainty,targets=targets);rec.log_metrics(registered_artifact_count=sum(f.is_file() for f in (Path(summary['recorder_local_dir'])/'artifacts').rglob('*')))
for name,value in [('backtest_benchmark_comparison',table),('annual_metrics',annual),('uncertainty',uncertainty),('training_audit',pd.DataFrame(audits)),('feature_usage',pd.DataFrame(selection))]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'segments':dataset.segments,'artifact_policy':policy}));a=Path(summary['recorder_local_dir'])/'artifacts';atomic_json({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');print(pd.DataFrame(selection).to_string(index=False));print(table[['family','slippage_bps','return_pct','monitored_drawdown_pct']].to_string(index=False))


# 原报告毛利率同比：固定训练回测未通过

完成6次真实拟合及6套新增Qlib原生账户。原全训练域3014主样本、20741辅助样本、3个固定seed/300轮末轮等权，小配置头、原标签/损失/平滑及全部执行风控保持原设定。双方共享可用性标记；候选唯一新增当季原报告GAAP毛利率减去年同季原报告毛利率，单位百分点。不用后来比较值回填，不改非GAAP，不剔除存货、出口限制或其他特殊费用。毛利率也不是纯粹的定价能力或市场超预期，变化可能来自产品结构、成本及会计口径。

48份已留存的原始SEC附件文本逐份解析GAAP表，百分号与单独表格列两种格式均支持，季度身份由原EPS表头交叉核对。先归一化已有日期字段，再按全部原文日期中最晚一天之后的首个Qlib交易日使用；首4季同比未知，其后值持续到下一季。此为条件性延迟已披露信息重建，不是首发反应模型、原始HTML字节或真实接收时钟认证。父来源资格没有被提升，不能用本次模型结果倒推来源认证。

候选数值通道零初始化，配对初始输出相同；同训练域与基础缩放器相同。1794个已知训练日不等于独立事件数。新字段经Qlib独立store写入及D API读回进入Dataset训练。毛利率不涉及股份分母，无拆股调整，也不含上一轮EPS数值。

回测2023-01-03信号至2026-10-09退出，944区间；10万美元、冻结IBKR Pro Tiered费率投影，0/2/5bp账户分别结算实际持仓、月度费用和风险状态。既有历史反复使用，不构成独立测试。

|账户|0bp净收益|2bp净收益|5bp净收益|
|---|---:|---:|---:|
|同可用标记对照|391.64%|386.86%|379.18%|
|加入毛利率同比|363.17%|358.44%|268.45%|
|原长期小配置头|342.52%|335.36%|348.06%|
|SOXX|387.75%|387.73%|387.70%|

2bp候选监控回撤27.41%，回撤及CVaR门通过；126日滚动正超额43.96%，未达70%。三成本均弱于配对标记对照，5bp弱于原长期强参考，净SOXX超额、年度、滚动及区块置信门失败。标记对照0bp虽稍超SOXX，2/5bp未超；不能将本轮较好的对照事后改成新候选宣称发现alpha。该标记同时改变基础输入维度与初始化，而且它在整个评价期恒定可用，不是当日新市场信息。

决定not_promoted_delayed_gross_margin。该固定毛利率同比方案关闭，不更换符号、会计口径、滞后或模型救结果，也不继续在同一发行人同一批财报中逐栏搜索盈利指标。EPS及毛利率的两次分别固定的配对诊断都没有支持替换强参考；这不证明所有财务信息无用，只说明现有单发行人历史实际值代理没有满足目标。下一需要有明确新增内容的市场预期或多发行人信息及独立验证，不能复活已关闭的预期/指引路线。生产未改，稳定净SOXX超额未完成。

共用日期函数从EPS代码抽出后，48个已封存EPS事件的全部字段重新计算与原Recorder零容差一致，没有重训或重跑旧账户。3项针对性测试通过；本轮notebook完成6训练/6原生账户和Qlib读回，NAS逐对象冷恢复哈希通过。真实分红、成交时延和全分钟回撤的既有研究边界保留。

48季度生成44个毛利率同比值，训练期涉及29次季度更新。

2bp年度候选 / SOXX：2023：36.27% / 56.60%；2024：29.77% / 20.97%；2025：52.60% / 46.80%；2026：69.88% / 75.38%。2026为不完整年度。
